# 📦 Long Châu Scraper — Bước 2: Cào Dữ Liệu Bronze (Bronze Ingestion)

Notebook này đọc các URL ở trạng thái `pending` trong file queue `url_queue_longchau.json`, bóc tách dữ liệu `__NEXT_DATA__`, đính kèm 5 trường Metadata Tầng Bronze (`source_name`, `trust_score=0.75`, `source_url`, `ingestion_timestamp`, `file_hash`) và lưu file raw JSON vào `bronze/longchau/`.

In [ ]:
import sys
import time
import json
import re
import random
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import requests
from tqdm.notebook import tqdm

from config import (
    BRONZE_DIR,
    MAX_RETRIES,
    QUEUE_FILE,
    REQUEST_DELAY_MAX,
    REQUEST_DELAY_MIN,
    SOURCE_NAME,
    TIMEOUT_SECONDS,
    TRUST_SCORE,
    USER_AGENTS,
)

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

In [ ]:
def load_queue():
    if not QUEUE_FILE.exists():
        print(f'❌ Queue file không tồn tại tại {QUEUE_FILE}. Hãy chạy notebook 01_discover_urls trước!')
        sys.exit(1)
    with open(QUEUE_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_queue(queue):
    with open(QUEUE_FILE, 'w', encoding='utf-8') as f:
        json.dump(queue, f, ensure_ascii=False, indent=2)

def get_headers():
    return {
        'User-Agent': random.choice(USER_AGENTS),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
    }

def clean_slug_filename(url: str, sku: str = '') -> str:
    slug = url.split('/')[-1].replace('.html', '')
    slug_clean = re.sub(r'[^a-zA-Z0-9_-]', '_', slug)
    url_hash = hashlib.sha256(url.encode('utf-8')).hexdigest()[:8]
    if sku:
        return f'{sku}_{slug_clean}_{url_hash}.json'
    return f'{slug_clean}_{url_hash}.json'

def fetch_product_data(url: str):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=TIMEOUT_SECONDS)
            if resp.status_code == 200:
                match = re.search(r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>', resp.text, re.DOTALL)
                if match:
                    raw_json_str = match.group(1)
                    data = json.loads(raw_json_str)
                    page_props = data.get('props', {}).get('pageProps', {})
                    return page_props, raw_json_str
                else:
                    return None, 'Không tìm thấy thẻ __NEXT_DATA__'
            elif resp.status_code == 404:
                return None, 'HTTP 404 Not Found'
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, str(e)
        time.sleep(attempt * 1.5)
    return None, 'Quá số lần thử lại'

def scrape_bronze(limit: int = None, retry_failed: bool = False):
    queue = load_queue()
    target_urls = [url for url, info in queue.items() if info.get('status') == 'pending' or (retry_failed and info.get('status') == 'failed')]

    if not target_urls:
        print('✨ Không có URL nào ở trạng thái pending trong Queue!')
        return

    if limit:
        target_urls = target_urls[:limit]

    print(f'🚀 Bắt đầu cào dữ liệu Bronze cho {len(target_urls)} sản phẩm (Limit: {limit or "Tất cả"})')
    print(f'📂 Lưu file tại: {BRONZE_DIR}')

    success_count = 0
    fail_count = 0
    pbar = tqdm(target_urls, desc='Scraping Bronze Zone', unit='doc')

    for i, url in enumerate(pbar):
        queue[url]['attempts'] = queue[url].get('attempts', 0) + 1
        page_props, raw_or_err = fetch_product_data(url)

        if page_props:
            product = page_props.get('product') or {}
            sku = product.get('sku', '')
            raw_data_str = json.dumps(page_props, ensure_ascii=False)
            file_hash = hashlib.sha256(raw_data_str.encode('utf-8')).hexdigest()
            ingestion_time = datetime.now(timezone.utc).isoformat()
            lineage_hash = hashlib.sha256(f'{url}_{file_hash}_{ingestion_time}'.encode('utf-8')).hexdigest()

            bronze_record = {
                'source_name': SOURCE_NAME,
                'trust_score': TRUST_SCORE,
                'source_url': url,
                'ingestion_timestamp': ingestion_time,
                'file_hash': file_hash,
                'lineage_hash': lineage_hash,
                'raw_data': page_props
            }

            filename = clean_slug_filename(url, sku)
            file_path = BRONZE_DIR / filename
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(bronze_record, f, ensure_ascii=False, indent=2)

            queue[url]['status'] = 'done'
            queue[url]['scraped_at'] = ingestion_time
            queue[url]['file_path'] = str(file_path)
            queue[url]['error'] = None
            success_count += 1
        else:
            queue[url]['status'] = 'failed'
            queue[url]['error'] = str(raw_or_err)
            fail_count += 1

        if (i + 1) % 5 == 0:
            save_queue(queue)
        time.sleep(random.uniform(REQUEST_DELAY_MIN, REQUEST_DELAY_MAX))

    save_queue(queue)
    print('\n==========================================')
    print('🎉 Hoàn thành phiên cào Bronze!')
    print(f'✅ Thành công: {success_count}')
    print(f'❌ Thất bại: {fail_count}')
    print(f'💾 Tổng số file tại Bronze: {len(list(BRONZE_DIR.glob("*.json")))}')


In [ ]:
# Chạy cào 10 sản phẩm mẫu (đặt limit=None để cào toàn bộ)
scrape_bronze(limit=10)